# Dadi vs Moments Parameter Comparison

Compares dadi's best-of-N estimate against moments' best-of-N estimate, per parameter,
for whichever simulations happen to have results from **both** engines (the two engines
don't necessarily have the same set of completed sims, so this only plots the overlap).

This is engine-vs-engine, not vs. ground truth: if the two independent inference
methods agree, points should fall near the y=x line regardless of whether either one
recovered the true parameter.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")

In [ ]:
# Configuration - UPDATE THESE PATHS FOR YOUR EXPERIMENT
EXPERIMENT_NAME = 'split_migration_growth_both'
BASE_PATH = Path(f'/projects/kernlab/akapoor/Infer_Demography/experiments/{EXPERIMENT_NAME}')
RUNS_PATH = BASE_PATH / 'runs'

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Runs path: {RUNS_PATH}")
print(f"Path exists: {RUNS_PATH.exists()}")

In [ ]:
def extract_parameters(fit_data):
    """Extract parameters and likelihood from a best_fit.pkl blob."""
    params = {}
    likelihood = None

    if isinstance(fit_data, dict):
        if 'param_order' in fit_data and 'opt_params' in fit_data:
            param_order = fit_data['param_order']
            param_values = fit_data['opt_params']
            if len(param_order) == len(param_values):
                params = dict(zip(param_order, param_values))
            if 'opt_ll' in fit_data:
                likelihood = fit_data['opt_ll']
        elif 'best_params' in fit_data:
            if isinstance(fit_data['best_params'], dict):
                params = fit_data['best_params']
            elif 'param_order' in fit_data:
                order = fit_data['param_order']
                values = fit_data['best_params']
                params = dict(zip(order, values))
            for key in ['best_ll', 'log_likelihood', 'll']:
                if key in fit_data:
                    val = fit_data[key]
                    likelihood = val[0] if isinstance(val, (list, tuple)) else val
                    break
        else:
            param_keys = [k for k in fit_data.keys() if not k.startswith('_') and k not in ['ll', 'log_likelihood', 'best_ll', 'opt_ll']]
            for key in param_keys:
                if isinstance(fit_data[key], (int, float)):
                    params[key] = fit_data[key]
            for key in ['best_ll', 'log_likelihood', 'll', 'opt_ll']:
                if key in fit_data:
                    val = fit_data[key]
                    likelihood = val[0] if isinstance(val, (list, tuple)) else val
                    break

    if likelihood is not None:
        params['log_likelihood'] = float(likelihood)
    return params


def load_engine_results(runs_path, engine):
    """Load best_fit.pkl for one engine across every run_{sim}_{rep} directory.
    Returns {sim_id: {rep_id: raw_blob}}.
    """
    results = {}
    for run_dir in sorted(runs_path.glob('run_*_*')):
        if not run_dir.is_dir():
            continue
        parts = run_dir.name.split('_')
        if len(parts) < 3:
            continue
        sim_id, rep_id = parts[1], parts[2]

        fit_file = run_dir / 'inferences' / engine / 'best_fit.pkl'
        if not fit_file.exists():
            continue
        try:
            with open(fit_file, 'rb') as f:
                data = pickle.load(f)
        except Exception as e:
            print(f"Error loading {fit_file}: {e}")
            continue

        results.setdefault(sim_id, {})[rep_id] = data

    print(f"Loaded {engine} results for {len(results)} simulations")
    return results


def results_to_long_df(results, engine):
    """Turn {sim: {rep: blob}} into a long dataframe: simulation, replicate, parameter, estimated."""
    rows = []
    for sim_id, reps in results.items():
        for rep_id, blob in reps.items():
            est_params = extract_parameters(blob)
            for param, val in est_params.items():
                rows.append({
                    'simulation': sim_id,
                    'replicate': rep_id,
                    'parameter': param,
                    'estimated': val,
                })
    df = pd.DataFrame(rows)
    print(f"{engine}: {len(df)} rows across {df['simulation'].nunique() if len(df) else 0} simulations")
    return df


def best_of_n(df):
    """Filter a long df down to the single best (highest log-likelihood) replicate per
    simulation. idxmax skips NaN by default, so a failed/NaN replicate can't win.
    """
    ll_rows = df[df['parameter'] == 'log_likelihood']
    if ll_rows.empty:
        return df.iloc[0:0]
    best_pairs = ll_rows.loc[ll_rows.groupby('simulation')['estimated'].idxmax(), ['simulation', 'replicate']]
    return df.merge(best_pairs, on=['simulation', 'replicate'], how='inner')


print("Helper functions defined!")

In [ ]:
# Load both engines independently -- they don't need the same set of completed sims
dadi_results = load_engine_results(RUNS_PATH, 'dadi')
moments_results = load_engine_results(RUNS_PATH, 'moments')

df_dadi_all = results_to_long_df(dadi_results, 'dadi')
df_moments_all = results_to_long_df(moments_results, 'moments')

df_dadi_best = best_of_n(df_dadi_all)
df_moments_best = best_of_n(df_moments_all)

print(f"\ndadi best-of-N: {df_dadi_best['simulation'].nunique()} simulations")
print(f"moments best-of-N: {df_moments_best['simulation'].nunique()} simulations")

In [ ]:
# Keep only (simulation, parameter) pairs present in BOTH engines' best-of-N results
merged = pd.merge(
    df_dadi_best[df_dadi_best['parameter'] != 'log_likelihood'][['simulation', 'parameter', 'estimated']].rename(columns={'estimated': 'dadi_estimate'}),
    df_moments_best[df_moments_best['parameter'] != 'log_likelihood'][['simulation', 'parameter', 'estimated']].rename(columns={'estimated': 'moments_estimate'}),
    on=['simulation', 'parameter'],
    how='inner',
)

n_sims_both = merged['simulation'].nunique()
n_sims_dadi_only = df_dadi_best['simulation'].nunique() - n_sims_both
n_sims_moments_only = df_moments_best['simulation'].nunique() - n_sims_both

print(f"Simulations with both dadi and moments results: {n_sims_both}")
print(f"Simulations with dadi only: {n_sims_dadi_only}")
print(f"Simulations with moments only: {n_sims_moments_only}")
merged.head(10)

In [ ]:
def plot_engine_comparison(merged_df, param_name):
    """Scatter dadi's best estimate against moments' best estimate for one parameter."""
    data = merged_df[merged_df['parameter'] == param_name].dropna(subset=['dadi_estimate', 'moments_estimate'])

    if len(data) == 0:
        print(f"No overlapping data for {param_name}")
        return None

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(data['moments_estimate'], data['dadi_estimate'], alpha=0.6, s=50, edgecolors='black', linewidth=0.5)

    lo = min(data['moments_estimate'].min(), data['dadi_estimate'].min())
    hi = max(data['moments_estimate'].max(), data['dadi_estimate'].max())
    ax.plot([lo, hi], [lo, hi], 'r--', alpha=0.7, label='dadi = moments')

    corr = data['moments_estimate'].corr(data['dadi_estimate'])
    ax.set_xlabel(f'moments {param_name}')
    ax.set_ylabel(f'dadi {param_name}')
    ax.set_title(f'{param_name}: dadi vs moments (best-of-N)\nCorr = {corr:.3f}, n = {len(data)}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"{param_name}: n={len(data)}, corr={corr:.4f}")
    return corr


parameters = [p for p in merged['parameter'].unique()]
print(f"Plotting {len(parameters)} parameters: {parameters}\n")

for param in parameters:
    plot_engine_comparison(merged, param)
    print("=" * 60)